In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset (Alzheimer's Disease)
# Prefer raw Excel with Subject ID (needed for participant splits).
# Skip cleaned CSVs that already dropped Subject ID.
# ----------------------------------------------------
_here = Path.cwd().resolve()
candidate_paths = []
for _root in [_here, *_here.parents]:
    candidate_paths.extend(
        [
            _root / "Datasets" / "Alzhimers.xlsx",
            _root / "Datasets" / "Alzheimer.xlsx",
            _root / "Alzhimers.xlsx",
            _root / "Alzheimer.xlsx",
        ]
    )
# Local copies last (often pre-cleaned without Subject ID)
candidate_paths.extend(
    [
        _here / "Alzhimers.xlsx",
        _here / "Alzheimer.xlsx",
        _here / "clean_alzheimer.csv",
        _here / ".." / "Other GANS" / "clean_alzheimer.csv",
        _here / ".." / ".." / "Other GANS" / "clean_alzheimer.csv",
        _here / ".." / ".." / ".." / "Datasets" / "Alzhimers.xlsx",
    ]
)

def _load_alzheimer_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    return pd.read_csv(path)

data_path = None
raw_data = None
for p in candidate_paths:
    p = Path(p)
    if not p.exists():
        continue
    df = _load_alzheimer_table(p)
    # Require Subject ID for participant-level evaluation
    if "Subject ID" not in df.columns:
        print(f"Skipping {p} (no 'Subject ID' column)")
        continue
    data_path = p
    raw_data = df
    break

if raw_data is None:
    raise FileNotFoundError(
        "Alzheimer dataset with 'Subject ID' not found. "
        "Expected Datasets/Alzhimers.xlsx under the repo root."
    )

print(f"Loading: {data_path}")

target_col = "Group"
subject_col = "Subject ID"

# Keep Subject ID for participant-level splits; drop other IDs / unused cols
ad_data = raw_data.drop(columns=["M/F", "MRI ID", "Hand"], errors="ignore")
if subject_col not in ad_data.columns:
    raise KeyError(f"{subject_col} is required for participant-level train/test splits")

ad_data[target_col] = ad_data[target_col].replace(
    {"Demented": 1, "Nondemented": 0, "Converted": 1}
)
ad_data[target_col] = pd.to_numeric(ad_data[target_col], errors="coerce")

# Complete-case only — no mean / median / mode imputation
_before = len(ad_data)
_n_missing_rows = int(ad_data.isna().any(axis=1).sum())
ad_data = ad_data.dropna().reset_index(drop=True)
ad_data[target_col] = ad_data[target_col].astype(int)
print(
    f"Dropped {_before - len(ad_data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not ad_data.isna().any().any(), "Unexpected NaNs remain after dropna"

def _participant_split(df, subject_col=subject_col, label_col=target_col, test_size=0.2, seed=42):
    """Split by Subject ID so all sessions of a participant stay in one fold."""
    subj_label = df.groupby(subject_col)[label_col].first()
    subjects = subj_label.index.to_numpy()
    subj_y = subj_label.to_numpy()
    subj_train, subj_test = train_test_split(
        subjects, test_size=test_size, random_state=seed, stratify=subj_y
    )
    assert set(subj_train).isdisjoint(set(subj_test)), "Subject overlap between train and test"
    train_df = (
        df[df[subject_col].isin(subj_train)]
        .drop(columns=[subject_col])
        .reset_index(drop=True)
    )
    test_df = (
        df[df[subject_col].isin(subj_test)]
        .drop(columns=[subject_col])
        .reset_index(drop=True)
    )
    print(
        f"Participant split: subjects={len(subjects)} "
        f"(train={len(subj_train)}, test={len(subj_test)}); "
        f"sessions train={len(train_df)}, test={len(test_df)}"
    )
    print("Train Group:", train_df[label_col].value_counts().to_dict())
    print("Test Group:", test_df[label_col].value_counts().to_dict())
    return train_df, test_df

# Placeholder until SEED/TEST_SIZE are set below; single-run cell re-applies split
alzheimer_data = ad_data.drop(columns=[subject_col]).copy()
X = alzheimer_data.drop(columns=[target_col])
y = alzheimer_data[target_col]
train_real = test_real = None

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(alzheimer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


Loading: /home/gopi.battineni/SYNTH_BENCHMARK/SYNTH/Datasets/Alzhimers.xlsx
Dropped 19 rows with missing feature values (complete-case; no imputation; rows_with_na=19)


In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

# Participant-level split (no subject leakage across train/test)
train_real, test_real = _participant_split(
    ad_data, subject_col=subject_col, label_col=target_col, test_size=TEST_SIZE, seed=seed
)
alzheimer_data = train_real.copy()  # generators fit on train subjects only
X = pd.concat([train_real, test_real], ignore_index=True).drop(columns=[target_col])
y = pd.concat([train_real, test_real], ignore_index=True)[target_col]

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)



================ SINGLE RUN ================
Participant split: subjects=142 (train=113, test=29); sessions train=284, test=70
Train Group: {0: 151, 1: 133}
Test Group: {0: 39, 1: 31}
Training TabDDPM...
[0]
12
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(12)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.4234 Sum: 0.4234
Step 1000/1000 MLoss: 0.0 GLoss: 0.347 Sum: 0.347
mlp
Sample timestep    0
Discrete cols: [0, 3, 4, 5]
Num shape:  (1000, 10)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 370.97it/s]|
Column Shapes Score: 41.03%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 255.53it/s]|
Column Pair Trends Score: 33.83%

Overall Score (Average): 37.43%

TabDDPM: 0.3743


In [4]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()

Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 291.38it/s]|
Column Shapes Score: 65.4%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 285.76it/s]|
Column Pair Trends Score: 31.32%

Overall Score (Average): 48.36%

ForestDiffusion: 0.4836


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col=None,
    models=None,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51],
    label=None,
    resplit=False,
):
    """Evaluate classifiers with Acc/F1/Precision/Recall.

    For Alzheimer participant splits, keep resplit=False so sessions stay in the
    fixed train_real / test_real folds (seed only varies classifier RNG).
    """
    if label is not None and label_col is None:
        label_col = label
    if label_col is None:
        raise ValueError("label_col is required")
    if models is None:
        raise ValueError("models is required")

    results = []
    train_df = train_df.copy()
    test_df = test_df.copy()
    # Drop Subject ID if still present
    for _df in (train_df, test_df):
        if "Subject ID" in _df.columns:
            _df.drop(columns=["Subject ID"], inplace=True)
    train_df[label_col] = pd.to_numeric(train_df[label_col], errors="coerce").astype(int)
    test_df[label_col] = pd.to_numeric(test_df[label_col], errors="coerce").astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]
            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if resplit:
                X_train, _, y_train, _ = train_test_split(
                    X_train,
                    y_train,
                    test_size=test_size,
                    random_state=seed,
                    stratify=y_train
                )
                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=y_test
                )

            # Complete-case residual drop (no imputation)
            train_keep = ~X_train.isna().any(axis=1)
            test_keep = ~X_test.isna().any(axis=1)
            X_train, y_train = X_train.loc[train_keep], y_train.loc[train_keep]
            X_test, y_test = X_test.loc[test_keep], y_test.loc[test_keep]

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = "Group"

model_order = ["TabDDPM", "ForestDiffusion"]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col="Group",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds,
    resplit=False,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not trained")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col="Group",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds,
        resplit=False,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
1,SVM-RBF,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
5,RandomForest,0.9771 ± 0.0070,0.9749 ± 0.0075,0.9511 ± 0.0144,1.0000 ± 0.0000
3,NaiveBayes,0.9714 ± 0.0000,0.9688 ± 0.0000,0.9394 ± 0.0000,1.0000 ± 0.0000
7,GradientBoost,0.9714 ± 0.0000,0.9688 ± 0.0000,0.9394 ± 0.0000,1.0000 ± 0.0000
6,ExtraTrees,0.9586 ± 0.0077,0.9554 ± 0.0079,0.9147 ± 0.0146,1.0000 ± 0.0000
8,AdaBoost,0.9571 ± 0.0000,0.9538 ± 0.0000,0.9118 ± 0.0000,1.0000 ± 0.0000
9,MLP,0.9429 ± 0.0111,0.9380 ± 0.0126,0.9020 ± 0.0116,0.9774 ± 0.0252
4,DecisionTree,0.9329 ± 0.0091,0.9291 ± 0.0096,0.8727 ± 0.0124,0.9935 ± 0.0129
2,KNN,0.9143 ± 0.0000,0.8929 ± 0.0000,1.0000 ± 0.0000,0.8065 ± 0.0000


TabDDPM - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.5571 ± 0.0000,0.5507 ± 0.0000,0.5000 ± 0.0000,0.6129 ± 0.0000
1,SVM-RBF,0.5571 ± 0.0000,0.5507 ± 0.0000,0.5000 ± 0.0000,0.6129 ± 0.0000
7,GradientBoost,0.5429 ± 0.0000,0.4667 ± 0.0000,0.4828 ± 0.0000,0.4516 ± 0.0000
8,AdaBoost,0.5286 ± 0.0000,0.4590 ± 0.0000,0.4667 ± 0.0000,0.4516 ± 0.0000
4,DecisionTree,0.5157 ± 0.0207,0.4543 ± 0.0138,0.4542 ± 0.0221,0.4548 ± 0.0097
9,MLP,0.4543 ± 0.0237,0.3289 ± 0.0418,0.3600 ± 0.0369,0.3032 ± 0.0461
6,ExtraTrees,0.4429 ± 0.0367,0.3196 ± 0.0472,0.3483 ± 0.0461,0.2968 ± 0.0516
5,RandomForest,0.4357 ± 0.0390,0.3327 ± 0.0590,0.3478 ± 0.0558,0.3194 ± 0.0620
3,NaiveBayes,0.4286 ± 0.0000,0.5833 ± 0.0000,0.4308 ± 0.0000,0.9032 ± 0.0000
2,KNN,0.4143 ± 0.0000,0.2545 ± 0.0000,0.2917 ± 0.0000,0.2258 ± 0.0000


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
1,SVM-RBF,0.9857 ± 0.0000,0.9841 ± 0.0000,0.9688 ± 0.0000,1.0000 ± 0.0000
3,NaiveBayes,0.9714 ± 0.0000,0.9688 ± 0.0000,0.9394 ± 0.0000,1.0000 ± 0.0000
6,ExtraTrees,0.9671 ± 0.0091,0.9643 ± 0.0096,0.9313 ± 0.0180,1.0000 ± 0.0000
7,GradientBoost,0.9571 ± 0.0000,0.9538 ± 0.0000,0.9118 ± 0.0000,1.0000 ± 0.0000
5,RandomForest,0.9571 ± 0.0000,0.9538 ± 0.0000,0.9118 ± 0.0000,1.0000 ± 0.0000
8,AdaBoost,0.9571 ± 0.0000,0.9538 ± 0.0000,0.9118 ± 0.0000,1.0000 ± 0.0000
9,MLP,0.9343 ± 0.0183,0.9291 ± 0.0190,0.8920 ± 0.0326,0.9710 ± 0.0304
4,DecisionTree,0.8786 ± 0.0258,0.8800 ± 0.0226,0.7865 ± 0.0362,1.0000 ± 0.0000
2,KNN,0.8714 ± 0.0000,0.8571 ± 0.0000,0.8438 ± 0.0000,0.8710 ± 0.0000


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,ForestDiffusion,LogReg,0.000000,0.000000,0.000000,0.000000,0.9857 ± 0.0000,0.9857 ± 0.0000
1,ForestDiffusion,SVM-RBF,0.000000,0.000000,0.000000,0.000000,0.9857 ± 0.0000,0.9857 ± 0.0000
2,ForestDiffusion,RandomForest,0.020000,0.021055,0.039372,0.000000,0.9771 ± 0.0070,0.9571 ± 0.0000
3,ForestDiffusion,NaiveBayes,0.000000,0.000000,0.000000,0.000000,0.9714 ± 0.0000,0.9714 ± 0.0000
4,ForestDiffusion,GradientBoost,0.014286,0.014904,0.027629,0.000000,0.9714 ± 0.0000,0.9571 ± 0.0000
5,ForestDiffusion,ExtraTrees,-0.008571,-0.008944,-0.016592,0.000000,0.9586 ± 0.0077,0.9671 ± 0.0091
6,ForestDiffusion,AdaBoost,0.000000,0.000000,0.000000,0.000000,0.9571 ± 0.0000,0.9571 ± 0.0000
7,ForestDiffusion,MLP,0.008571,0.008847,0.009927,0.006452,0.9429 ± 0.0111,0.9343 ± 0.0183
8,ForestDiffusion,DecisionTree,0.054286,0.049114,0.086221,-0.006452,0.9329 ± 0.0091,0.8786 ± 0.0258
9,ForestDiffusion,KNN,0.042857,0.035714,0.156250,-0.064516,0.9143 ± 0.0000,0.8714 ± 0.0000


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.013143,0.012069,0.030281,-0.006452
1,TabDDPM,0.472000,0.524936,0.518637,0.514516


In [9]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
